In [ ]:
import os
import sys
import json
import pandas as pd
from collections import Counter

ROOT_DIR = "/home/temari/god please no diploma/restore_punct/"
DATA_DIR = os.path.join(ROOT_DIR, "data")

try:
    from extract_labels_standardized import ID_TO_LABEL
except ImportError:
    sys.path.insert(0, ROOT_DIR)
    from extract_labels_standardized import ID_TO_LABEL

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"Loaded {len(ID_TO_LABEL)} punctuation labels")

ROOT_DIR: /home/temari/god please no diploma/restore_punct/
DATA_DIR: /home/temari/god please no diploma/restore_punct/data
Loaded 28 punctuation labels


In [ ]:
train_all_path = os.path.join(DATA_DIR, "train_all.json")
train_all_synthetic_path = os.path.join(DATA_DIR, "train_all_synthetic.json")

print("Loading datasets...")
with open(train_all_path, 'r', encoding='utf-8') as f:
    train_all = json.load(f)

with open(train_all_synthetic_path, 'r', encoding='utf-8') as f:
    train_all_synthetic = json.load(f)

print(f"✓ train_all.json: {len(train_all)} examples loaded")
print(f"✓ train_all_synthetic.json: {len(train_all_synthetic)} examples loaded")

Loading datasets...
✓ train_all.json: 61243 examples loaded
✓ train_all_synthetic.json: 66197 examples loaded


In [ ]:
punct_counter_train_all = Counter()

for example in train_all:
    for label_id in example["ner_tags"]:
        if label_id != 0:
            punct_label = ID_TO_LABEL.get(label_id, f"UNKNOWN_{label_id}")
            punct_counter_train_all[punct_label] += 1

top_3_most = punct_counter_train_all.most_common(3)
top_3_rarest = punct_counter_train_all.most_common()[-3:][::-1]

print("\n=== PUNCTUATION DISTRIBUTION (train_all.json) ===")
print("\nTop-3 Most Frequent:")
for punct, count in top_3_most:
    print(f"  {punct!r}: {count}")

print("\nTop-3 Rarest:")
for punct, count in top_3_rarest:
    print(f"  {punct!r}: {count}")

print(f"\nTotal unique punctuation marks: {len(punct_counter_train_all)}")
print(f"Total punctuation occurrences: {sum(punct_counter_train_all.values())}")


=== PUNCTUATION DISTRIBUTION (train_all.json) ===

Top-3 Most Frequent:
  ',': 906621
  '.': 614422
  '"': 290361

Top-3 Rarest:
  '"!': 70
  '"?': 256
  '! -': 334

Total unique punctuation marks: 27
Total punctuation occurrences: 2177192


In [5]:
# Create table: Top-3 and Rarest Punctuation Signs
top_3_data = []

for rank, (punct, count) in enumerate(top_3_most, 1):
    top_3_data.append({
        "Rank": rank,
        "Punctuation Sign": punct,
        "Count": count,
        "Type": "Most Frequent"
    })

for rank, (punct, count) in enumerate(top_3_rarest, 1):
    top_3_data.append({
        "Rank": rank,
        "Punctuation Sign": punct,
        "Count": count,
        "Type": "Rarest"
    })

df_top3 = pd.DataFrame(top_3_data)

print("\n" + "="*70)
print("TABLE 1: TOP-3 MOST FREQUENT & TOP-3 RAREST PUNCTUATION SIGNS")
print("="*70)
print(df_top3.to_string(index=False))
print("="*70)


TABLE 1: TOP-3 MOST FREQUENT & TOP-3 RAREST PUNCTUATION SIGNS
 Rank Punctuation Sign  Count          Type
    1                , 906621 Most Frequent
    2                . 614422 Most Frequent
    3                " 290361 Most Frequent
    1               "!     70        Rarest
    2               "?    256        Rarest
    3              ! -    334        Rarest


In [ ]:
chunks_train_all = len(train_all)
chunks_train_all_synthetic = len(train_all_synthetic)
added_synthetic = chunks_train_all_synthetic - chunks_train_all

total_tokens_train_all = sum(len(ex["tokens"]) for ex in train_all)
total_tokens_synthetic = sum(len(ex["tokens"]) for ex in train_all_synthetic)

avg_tokens_train_all = total_tokens_train_all / chunks_train_all if chunks_train_all > 0 else 0
avg_tokens_synthetic = total_tokens_synthetic / chunks_train_all_synthetic if chunks_train_all_synthetic > 0 else 0

print("\n" + "="*70)
print("TABLE 2: CHUNK COUNT SUMMARY")
print("="*70)

chunk_summary = pd.DataFrame({
    "Dataset": ["train_all.json", "train_all_synthetic.json", "Difference (synthetic added)"],
    "Chunks": [chunks_train_all, chunks_train_all_synthetic, added_synthetic],
    "Total Tokens": [total_tokens_train_all, total_tokens_synthetic, 
                     total_tokens_synthetic - total_tokens_train_all],
    "Avg Tokens/Chunk": [f"{avg_tokens_train_all:.2f}", f"{avg_tokens_synthetic:.2f}", "-"]
})

print(chunk_summary.to_string(index=False))
print("="*70)


TABLE 2: CHUNK COUNT SUMMARY
                     Dataset  Chunks  Total Tokens Avg Tokens/Chunk
              train_all.json   61243      11487730           187.58
    train_all_synthetic.json   66197      11815923           178.50
Difference (synthetic added)    4954        328193                -


In [ ]:
punct_counter_synthetic = Counter()

for example in train_all_synthetic:
    for label_id in example["ner_tags"]:
        if label_id != 0:
            punct_label = ID_TO_LABEL.get(label_id, f"UNKNOWN_{label_id}")
            punct_counter_synthetic[punct_label] += 1

all_puncts = set(punct_counter_train_all.keys()) | set(punct_counter_synthetic.keys())
all_puncts_sorted = sorted(all_puncts)

comparison_data = []
for punct in all_puncts_sorted:
    count_train_all = punct_counter_train_all[punct]
    count_synthetic = punct_counter_synthetic[punct]
    total_count = count_train_all + count_synthetic
    
    pct_train_all = (count_train_all / total_count * 100) if total_count > 0 else 0
    pct_synthetic = (count_synthetic / total_count * 100) if total_count > 0 else 0
    
    comparison_data.append({
        "Punctuation": punct,
        "train_all": count_train_all,
        "synthetic": count_synthetic,
        "Total": total_count,
        "% in train_all": f"{pct_train_all:.1f}%",
        "% synthetic": f"{pct_synthetic:.1f}%"
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*100)
print("TABLE 3: COMPREHENSIVE PUNCTUATION STATISTICS (All Signs)")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

total_punct_train_all = sum(punct_counter_train_all.values())
total_punct_synthetic = sum(punct_counter_synthetic.values())
total_punct_combined = total_punct_train_all + total_punct_synthetic

print(f"\nSummary:")
print(f"  Total punctuation marks in train_all: {total_punct_train_all}")
print(f"  Total punctuation marks in synthetic: {total_punct_synthetic}")
print(f"  Combined total: {total_punct_combined}")
print(f"  Synthetic contribution: {total_punct_synthetic / total_punct_combined * 100:.2f}%")


TABLE 3: COMPREHENSIVE PUNCTUATION STATISTICS (All Signs)
Punctuation  train_all  synthetic   Total % in train_all % synthetic
          !       3530       4029    7559          46.7%       53.3%
        ! -        334       3785    4119           8.1%       91.9%
         !"        731        880    1611          45.4%       54.6%
       !" -        353       1075    1428          24.7%       75.3%
          "     290361     290652  581013          50.0%       50.0%
        " -       4907       4929    9836          49.9%       50.1%
         "!         70         70     140          50.0%       50.0%
         ""       1067       1067    2134          50.0%       50.0%
         ",      33656      33740   67396          49.9%       50.1%
       ", -      29607      30093   59700          49.6%       50.4%
         ".      50652      51902  102554          49.4%       50.6%
         "?        256        261     517          49.5%       50.5%
          ,     906621     939953 1846574   